# Metadata for ablation

1. Use old code from `scripts/one_beat.py` to find heartrate for each subject.
2. Keep the filters from the old code
3. Explore heart rates of the subjects
4. Run `one_beat.py` on all the files from Zenodo before running this notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Run one_beat.py to get the summary dataframes - use all the files from Zenodo
# Read all the dataframes and combine them into one summary dataframe

hr_dataframes = []
for i in range(1, 17):
    df_summary = pd.read_csv(f'../data/peak_summary_{i}.csv')
    df_hr = df_summary.groupby('subject')[['hr', 'exam_id', 'retain_subject']].first().reset_index()
    df_hr.loc[:, 'file_num'] = i
    hr_dataframes.append(df_hr)
df_hr = pd.concat(hr_dataframes, ignore_index=True)
df_hr.head()


In [ ]:
df_hr = df_hr[df_hr['retain_subject'] == True]

In [ ]:
# make a histogram of heart rates
# remove all the subject with heart rate above 150 bpm
hr_filter = (df_hr['hr'] <= 200)
plt.figure(figsize=(10, 6))
plt.hist(df_hr[hr_filter]['hr'], bins=20, color='skyblue', edgecolor='black')
plt.title('Distribution of Heart Rates')
plt.xlabel('Heart Rate (bpm)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
df_hr.shape

In [ ]:
# 10 seconds are represented in 4096 pixels
# So we only have 3 significant digits in the the heart rate values


In [ ]:
# find how many non-unique values of hr are there
digits = 3
df_hr['hr'].round(digits).value_counts()[df_hr['hr'].round(digits).value_counts() > 1]

In [ ]:
df_meta = pd.read_csv('../data/exams.csv')
df_meta.head()

In [ ]:
df_hr = df_hr.merge(
    df_meta[['exam_id', 'age', 'is_male', 'normal_ecg', 'nn_predicted_age']],
    on='exam_id',
    how='inner'
)

In [ ]:
df_hr.head()

In [ ]:
data_filter = df_hr['normal_ecg']
df_hr[data_filter]['hr'].hist()
plt.show()

In [ ]:
df_hr['hr_round'] = df_hr['hr'].round(3)
hr_id_dict = {}
for i, hr in enumerate(df_hr['hr_round'].value_counts().index):
    hr_id_dict[hr] = i

df_hr.loc[:, 'hr_id'] = df_hr['hr_round'].map(hr_id_dict)

In [ ]:
df_hr[
    (df_hr['hr_id'] == 11026)
]

In [ ]:
avg_trace_16 = np.load('../data/one_beat_array_16.npy')
avg_trace_16.shape

In [ ]:
df_hr[df_hr['file_num'] == 16]['hr_id'].value_counts().head()

In [ ]:
df_hr[
    (df_hr['file_num'] == 16) & (df_hr['hr_id'] == 125)
]

In [ ]:
df_summary_16 = pd.read_csv('../data/peak_summary_16.csv')
df_summary_16.head()

In [ ]:
df_hr_16 = df_hr[df_hr['file_num'] == 16].reset_index(drop=True)
some_hr_16_indices = df_hr_16[df_hr_16['hr_id'] == 125].index.values

In [ ]:
chan = 2
start = 1800
end = 2330
plt.figure(figsize=(10, 6))
for index in some_hr_16_indices:
    x = np.arange(start, end)
    plt.plot(x, avg_trace_16[index, start:end ,chan], label=f"Subejct {index}")
plt.legend()
plt.show()

In [ ]:
some_hr_16_indices = df_hr_16[df_hr_16['hr_id'] == 244].index.values
chan = 2
start, end = 1800, 2330
plt.figure(figsize=(10, 6))
for index in some_hr_16_indices:
    x = np.arange(start, end)
    plt.plot(x, avg_trace_16[index, start:end ,chan], label=f"Subejct {index}")
plt.legend()
plt.show()

In [ ]:
df_hr_16[df_hr_16['hr_id'].isin([125, 244])].sort_values(by='hr_id')